# Chatterbox TTS Server — AMD ROCm GPU 部署指南

> **平台：** AMD OneClick Cloud (radeon.anruicloud.com) · ROCm 7.2+ · RX 7900 / RDNA3 (gfx1100)

## 简介

本 Notebook 在 AMD GPU 云平台上部署并运行 [Chatterbox TTS Server](https://github.com/devnen/Chatterbox-TTS-Server)，这是一个基于 Chatterbox 模型的自托管文字转语音服务器。

**Chatterbox TTS Server 功能：**
- Web UI 界面 + OpenAI 兼容 REST API
- 28 种预置声音
- 声音克隆（上传参考音频）
- 大规模有声书处理
- AMD ROCm GPU 加速

## 你会学到什么

- 如何在云端 Jupyter 环境中验证 AMD GPU / ROCm 状态
- 如何使用 RDNA4 兼容的 ROCm 7.2 PyTorch 环境安装 TTS 服务
- 如何在 Notebook 中以后台进程运行 TTS 服务器
- 如何通过 OpenAI 兼容 API 生成语音
- 如何在 Jupyter 中构建交互式 TTS 控制界面

## 环境要求

| 项目 | 要求 |
|---|---|
| GPU | AMD RX 7900 / RDNA3 (gfx1100) 或 RX 9070 / RDNA4 (gfx1201) |
| ROCm | 7.0+ |
| Python | 3.10 – 3.12 |
| 磁盘 | ~5 GB（模型缓存） |
| 网络 | 可访问 hf-mirror.com（HuggingFace 镜像） |

---

## 第一步 — 验证 AMD GPU 与 ROCm 环境

In [1]:
import subprocess, sys

print("=" * 60)
print("Python 版本:", sys.version)
print("=" * 60)

# 查看 ROCm GPU 信息
rocm = subprocess.run(["rocminfo"], capture_output=True, text=True)
for line in rocm.stdout.splitlines():
    if any(k in line for k in ["Name", "Marketing", "gfx", "Vendor"]):
        print(line.strip())

# GPU 状态概览
print("\n--- rocm-smi ---")
smi = subprocess.run(["rocm-smi"], capture_output=True, text=True)
print(smi.stdout[:1000] if smi.stdout else smi.stderr[:500])

Python 版本: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
Name:                    AMD EPYC 9334 32-Core Processor
Marketing Name:          AMD EPYC 9334 32-Core Processor
Vendor Name:             CPU
Name:                    AMD EPYC 9334 32-Core Processor
Marketing Name:          AMD EPYC 9334 32-Core Processor
Vendor Name:             CPU
Name:                    gfx1100
Marketing Name:          AMD Radeon Graphics
Vendor Name:             AMD
Name:                    amdgcn-amd-amdhsa--gfx1100
Name:                    amdgcn-amd-amdhsa--gfx11-generic

--- rocm-smi ---


======================================== ROCm System Management Interface ========================================
================================================== Concise Info ==================================================
Device  Node  IDs              Temp    Power  Partitions          SCLK  MCLK   Fan    Perf  PwrCap  VRAM%  GPU%  
              (DID,     GUID)  (Edge)  (Avg)  (Mem, Compute, ID)        

In [2]:
import torch

print("PyTorch 版本:", torch.__version__)
print("ROCm 可用:  ", torch.cuda.is_available())
print("GPU 数量:   ", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU 型号:   ", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"显存总量:    {props.total_memory / 1024**3:.1f} GB")
else:
    print("\n⚠️  未检测到 ROCm GPU，请检查驱动安装。")

PyTorch 版本: 2.9.1+gitff65f5b
ROCm 可用:   True
GPU 数量:    1
GPU 型号:    AMD Radeon Graphics
显存总量:    48.0 GB


## 第二步 — 配置 HuggingFace 镜像

AMD OneClick Cloud 平台无法直接访问 huggingface.co，使用 [hf-mirror.com](https://hf-mirror.com) 镜像下载模型。

In [2]:
import os, urllib.request

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

try:
    urllib.request.urlopen("https://hf-mirror.com", timeout=10)
    print("✅ hf-mirror.com 可访问")
except Exception as e:
    print(f"⚠️  镜像站无法访问: {e}")

✅ hf-mirror.com 可访问


## 第三步 — 获取 Chatterbox TTS Server 仓库

In [3]:
import os, subprocess

REPO_DIR = "/workspace/repo"

if os.path.exists(os.path.join(REPO_DIR, "server.py")):
    print(f"✅ 仓库已存在：{REPO_DIR}")
else:
    print("正在克隆仓库...")
    result = subprocess.run(
        ["git", "clone", "https://github.com/devnen/Chatterbox-TTS-Server.git", REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"✅ 克隆完成：{REPO_DIR}")
    else:
        print("❌ 克隆失败:", result.stderr)

print("\n仓库文件列表:")
print([f for f in os.listdir(REPO_DIR) if not f.startswith('.')])

✅ 仓库已存在：/workspace/repo

仓库文件列表:
['config.yaml', 'venv', 'README.md', 'Dockerfile.rocm', 'chatterbox-v2-add-disc.zip', 'Chatterbox_TTS_Colab_Demo.ipynb', 'requirements.txt', 'docker-compose-cu130.yml', 'engine.py', 'docker-compose-cpu.yml', 'Untitled.ipynb', 'requirements-nvidia.txt', 'docker-compose-strixhalo.yml', 'requirements-rdna4-init.txt', 'Dockerfile.cu128', 'requirements-rocm.txt', 'LICENSE', 'README_CUDA128.md', '__pycache__', 'docker-compose-cu128.yml', 'Dockerfile.cu130', 'chatterbox-rocm-amd.ipynb', 'download_model.py', 'requirements-nvidia-cu130.txt', 'docker-compose.yml', 'Dockerfile', 'README_Colab.md', 'docker-compose-rocm.yml', 'model_cache', 'static', 'start.py', 'requirements-rocm-init.txt', 'ui', 'logs', 'utils.py', 'Untitled1.ipynb', 'requirements-rdna4.txt', 'Untitled2-Copy1.ipynb', 'reference_audio', 'requirements-strixhalo.txt', 'requirements-nvidia-cu128.txt', 'models.py', 'docker-compose-rdna4.yml', 'voices', 'Dockerfile.cpu', 'server.py', 'start.bat', 'start

## 第四步 — 安装依赖

使用 RDNA4 兼容路径安装（适用于 ROCm 7.0+ 环境），分三步执行：

1. `requirements-rdna4-init.txt` — 安装 `torch 2.9.1+rocm7.2.3` PyTorch 轮子
2. `requirements-rdna4.txt` — 安装服务器其余依赖
3. `chatterbox-v2` + `s3tokenizer` + `onnx` — 安装 TTS 核心引擎

In [4]:
import subprocess, sys

def pip_install(args, cwd=None, label=""):
    """执行 pip 安装并显示结果"""
    if label:
        print(f"\n{'='*50}")
        print(f"📦 {label}")
        print('='*50)
    cmd = [sys.executable, "-m", "pip", "install"] + args
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=cwd)
    if result.returncode == 0:
        lines = [l for l in result.stdout.strip().splitlines() if l.strip()]
        print("\n".join(lines[-5:]) if lines else "（无输出）")
        print("✅ 完成")
    else:
        print("❌ 失败:")
        print(result.stderr[-2000:])
    return result.returncode

# # 步骤 1：安装 ROCm 7.2.3 PyTorch 环境
# pip_install(
#     ["-r", "requirements-rdna4-init.txt", "-q"],
#     cwd=REPO_DIR,
#     label="步骤 1/3：安装 PyTorch ROCm 7.2.3 环境"
# )

# 基础镜像已包含 PyTorch 2.9 + ROCm，无需重复安装
import torch
print(f"✅ PyTorch 已就绪: {torch.__version__}")
print(f"   ROCm 可用: {torch.cuda.is_available()}")
print(f"   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '未检测到'}")

✅ PyTorch 已就绪: 2.9.1+gitff65f5b
   ROCm 可用: True
   GPU: AMD Radeon Graphics


In [5]:
# 步骤 2：安装服务器依赖 等待完成
pip_install(
    ["-r", "requirements-rdna4.txt", "-q"],
    cwd=REPO_DIR,
    label="步骤 2/3：安装服务器依赖"
)


📦 步骤 2/3：安装服务器依赖
（无输出）
✅ 完成


0

In [6]:
import subprocess, sys, os, zipfile

print("=" * 50)
print("📦 步骤 3/3：安装 chatterbox-v2 引擎")
print("=" * 50)

EXTRA_PKGS = ["protobuf>=3.20.1", "s3tokenizer==0.3.0", "onnx==1.16.0"]

def try_git_install():
    """方法 A：从 GitHub 直接安装"""
    print("尝试方法 A：git+https 安装...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-deps",
         "git+https://github.com/devnen/chatterbox-v2.git@master"] + EXTRA_PKGS + ["-q"],
        capture_output=True, text=True, timeout=180
    )
    return result.returncode == 0

def try_zip_install():
    """方法 B：从本地 zip 安装（GitHub 无法访问时使用）"""
    print("方法 A 失败（无法访问 GitHub），尝试方法 B：本地 zip 安装...")
    zip_candidates = [
        os.path.join(REPO_DIR, "chatterbox-v2-add-disc.zip"),
        "/workspace/chatterbox-v2.zip",
        "/tmp/chatterbox-v2.zip",
    ]
    zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)

    if not zip_path:
        print("❌ 未找到本地 zip 文件。")
        print("   请从本地电脑下载：https://github.com/devnen/chatterbox-v2/archive/master.zip")
        print("   上传到 /workspace/ 后重新运行此单元格。")
        return False

    print(f"   找到 zip：{zip_path}")
    extract_dir = "/tmp/chatterbox-v2-src"
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)

    # 找到含 pyproject.toml 或 setup.py 的目录
    install_dir = extract_dir
    for root, dirs, files in os.walk(extract_dir):
        if "pyproject.toml" in files or "setup.py" in files:
            install_dir = root
            break

    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-deps", install_dir] + EXTRA_PKGS + ["-q"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("❌ 安装失败:", result.stderr[-500:])
    return result.returncode == 0

# 执行安装
ok = try_git_install() or try_zip_install()

if ok:
    print("\n验证导入...")
    check = subprocess.run(
        [sys.executable, "-c",
         "from chatterbox.tts import ChatterboxTTS; print('✅ chatterbox 导入成功')"],
        capture_output=True, text=True
    )
    print(check.stdout or check.stderr)
else:
    print("\n⚠️  安装未完成，请按上述提示处理后重试。")

📦 步骤 3/3：安装 chatterbox-v2 引擎
尝试方法 A：git+https 安装...
方法 A 失败（无法访问 GitHub），尝试方法 B：本地 zip 安装...
   找到 zip：/workspace/repo/chatterbox-v2-add-disc.zip

验证导入...
✅ chatterbox 导入成功



## 第五步 — 配置服务器参数

In [7]:
import yaml, os

config_path = os.path.join(REPO_DIR, "config.yaml")

with open(config_path) as f:
    config = yaml.safe_load(f)

engine = config.setdefault("tts_engine", {})
engine["model_selector"] = "chatterbox"   # chatterbox-turbo 需要单独安装
engine["device"]         = "cuda"          # ROCm 通过 CUDA API 暴露
engine["bf16"]           = False           # 如显存充裕可改为 True 提升性能

with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("✅ config.yaml 已更新")
print(f"   model_selector : {engine['model_selector']}")
print(f"   device         : {engine['device']}")
print(f"   bf16           : {engine['bf16']}")

✅ config.yaml 已更新
   model_selector : chatterbox
   device         : cuda
   bf16           : False


## 第六步 — 启动 TTS 服务器

服务器以后台子进程方式运行。首次启动将从 hf-mirror.com 下载 Chatterbox 模型（约 2-3 GB），需等待数分钟。

In [8]:
import subprocess, os, time

LOG_FILE = "/tmp/chatterbox_tts.log"

# 停止已有进程
subprocess.run(["pkill", "-f", "server.py"], capture_output=True)
time.sleep(2)

env = {
    **os.environ,
    "HF_ENDPOINT"     : "https://hf-mirror.com",
    "PYTHONUNBUFFERED" : "1",
    # 如遇 HIP 错误，取消下行注释
    # "HSA_OVERRIDE_GFX_VERSION": "11.0.0",
}

proc = subprocess.Popen(
    ["python", "server.py"],
    cwd=REPO_DIR,
    env=env,
    stdout=open(LOG_FILE, "w"),
    stderr=subprocess.STDOUT,
)

print(f"✅ 服务器已启动（PID: {proc.pid}）")
print(f"   日志文件：{LOG_FILE}")
print("   正在等待模型下载和加载（首次运行约 3-10 分钟）...")

✅ 服务器已启动（PID: 1520）
   日志文件：/tmp/chatterbox_tts.log
   正在等待模型下载和加载（首次运行约 3-10 分钟）...


In [9]:
# 监控启动进度 — 每隔 15 秒检查一次，最多等待 10 分钟
import time, requests

def server_ready():
    try:
        return requests.get("http://localhost:8004/api/model-info", timeout=3).status_code == 200
    except:
        return False

print("等待服务器就绪...\n")
for i in range(40):
    if server_ready():
        print(f"\n✅ 服务器就绪！（用时约 {i*15} 秒）")
        break
    try:
        with open(LOG_FILE) as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]
            if lines:
                print(f"[{i*15:3d}s] {lines[-1][:110]}")
    except:
        pass
    time.sleep(15)
else:
    print("\n⚠️  超时，服务器未能在 10 分钟内启动，查看完整日志：")
    with open(LOG_FILE) as f:
        print(f.read()[-3000:])

等待服务器就绪...


✅ 服务器就绪！（用时约 0 秒）


## 第七步 — 验证 GPU 加速状态

In [10]:
import requests, subprocess

# 查询模型信息
info = requests.get("http://localhost:8004/api/model-info").json()
print("模型信息：")
for k, v in info.items():
    print(f"  {k}: {v}")

# 从日志确认 GPU 加速
print("\n--- 启动日志关键信息 ---")
with open("/tmp/chatterbox_tts.log") as f:
    for line in f:
        if any(k in line for k in ["device", "cuda", "ROCm", "GPU", "BF16", "loaded", "Model"]):
            print(line.strip())

模型信息：
  loaded: True
  type: original
  class_name: ChatterboxTTS
  device: cuda
  sample_rate: 24000
  supports_paralinguistic_tags: False
  available_paralinguistic_tags: []
  turbo_available_in_package: False
  multilingual_available_in_package: False
  supports_multilingual: False
  supported_languages: {'en': 'English'}

--- 启动日志关键信息 ---
2026-06-10 09:46:04 [INFO] server: Configuration loaded. Log file at: /workspace/repo/logs/tts_server.log
2026-06-10 09:46:04 [INFO] engine: Final device selection: cuda
2026-06-10 09:46:04 [INFO] engine: BF16 optimization: disabled (TTS_BF16=off)
2026-06-10 09:46:04 [INFO] engine: Model selector from config: 'chatterbox'
2026-06-10 09:46:04 [INFO] engine: Model selector 'chatterbox' resolved to Original model (ChatterboxTTS)
2026-06-10 09:46:04 [INFO] engine: Initializing ChatterboxTTS on device 'cuda'...
2026-06-10 09:46:04 [INFO] engine: Model type: original
loaded PerthNet (Implicit) at step 250,000
2026-06-10 09:46:17 [INFO] engine: BF16 opti

## 第八步 — 通过 API 生成语音

服务器提供 OpenAI 兼容的 `/v1/audio/speech` 接口。

In [11]:
import requests

# 获取全部可用声音
voices = requests.get("http://localhost:8004/v1/audio/voices").json().get("voices", [])
print(f"可用声音（共 {len(voices)} 个）：")
for i, v in enumerate(voices):
    print(f"  {i+1:2d}. {v}")

可用声音（共 28 个）：
   1. Abigail.wav
   2. Adrian.wav
   3. Alexander.wav
   4. Alice.wav
   5. Austin.wav
   6. Axel.wav
   7. Connor.wav
   8. Cora.wav
   9. Elena.wav
  10. Eli.wav
  11. Emily.wav
  12. Everett.wav
  13. Gabriel.wav
  14. Gianna.wav
  15. Henry.wav
  16. Ian.wav
  17. Jade.wav
  18. Jeremiah.wav
  19. Jordan.wav
  20. Julian.wav
  21. Layla.wav
  22. Leonardo.wav
  23. Michael.wav
  24. Miles.wav
  25. Olivia.wav
  26. Ryan.wav
  27. Taylor.wav
  28. Thomas.wav


In [12]:
import requests, IPython.display as ipd

TEXT  = "Hello! This is Chatterbox TTS running on AMD GPU with ROCm acceleration."
VOICE = "Alice.wav"   # 可改为上方列表中的任意声音

response = requests.post("http://localhost:8004/v1/audio/speech", json={
    "model": "tts-1",
    "input": TEXT,
    "voice": VOICE,
}, timeout=120)

print(f"状态码：{response.status_code}  |  音频大小：{len(response.content):,} 字节")

if response.status_code == 200:
    ipd.display(ipd.Audio(response.content, autoplay=True))
else:
    print("错误：", response.text)

状态码：200  |  音频大小：293,804 字节


## 第九步 — 交互式 TTS 控制界面

基于 `ipywidgets` 构建的完整交互界面，无需暴露任何端口。

In [13]:
import ipywidgets as widgets
import requests, IPython.display as ipd, os

# 动态获取声音列表
try:
    voices = requests.get("http://localhost:8004/v1/audio/voices").json()["voices"]
except:
    voices = ["Alice.wav", "Ryan.wav", "Emily.wav"]

# --- 控件定义 ---
title = widgets.HTML("<h3 style='margin:0;color:#d00'>🎙️ Chatterbox TTS — AMD ROCm</h3>")

text_input = widgets.Textarea(
    value="欢迎使用 Chatterbox TTS，运行于 AMD GPU 云平台。",
    placeholder="输入要合成的文本...",
    layout=widgets.Layout(width="100%", height="100px")
)

voice_dd = widgets.Dropdown(
    options=voices,
    value=voices[0] if voices else None,
    description="声音：",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="320px")
)

temperature_sl = widgets.FloatSlider(
    value=0.8, min=0.1, max=1.5, step=0.05,
    description="随机性：",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="420px")
)

exaggeration_sl = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05,
    description="夸张度：",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="420px")
)

generate_btn = widgets.Button(description="生成语音", button_style="primary", icon="play")
save_btn     = widgets.Button(description="保存 WAV", button_style="success",  icon="download", disabled=True)
status_lbl   = widgets.Label(value="就绪")
output_area  = widgets.Output()

_last_audio = {"data": None}

def on_generate(b):
    generate_btn.disabled = True
    status_lbl.value = "⏳ 生成中..."
    with output_area:
        output_area.clear_output(wait=True)
        try:
            r = requests.post(
                "http://localhost:8004/v1/audio/speech",
                json={"model": "tts-1", "input": text_input.value, "voice": voice_dd.value},
                timeout=120
            )
            if r.status_code == 200:
                _last_audio["data"] = r.content
                save_btn.disabled = False
                status_lbl.value = f"✅ 完成 — {len(r.content):,} 字节"
                ipd.display(ipd.Audio(r.content, autoplay=True))
            else:
                status_lbl.value = f"❌ 错误 {r.status_code}"
                print(r.text)
        except Exception as e:
            status_lbl.value = f"❌ {e}"
    generate_btn.disabled = False

def on_save(b):
    if _last_audio["data"]:
        path = "/workspace/output_tts.wav"
        with open(path, "wb") as f:
            f.write(_last_audio["data"])
        status_lbl.value = f"💾 已保存至 {path}"

generate_btn.on_click(on_generate)
save_btn.on_click(on_save)

# 布局
ui = widgets.VBox([
    title,
    widgets.HTML("<hr/>"),
    widgets.Label("合成文本："),
    text_input,
    widgets.HBox([voice_dd]),
    temperature_sl,
    exaggeration_sl,
    widgets.HBox([generate_btn, widgets.HTML("&nbsp;"), save_btn]),
    status_lbl,
    output_area,
], layout=widgets.Layout(padding="16px", border="1px solid #ccc", border_radius="8px", width="600px"))

ipd.display(ui)

## 第十步 — 批量生成语音

In [ ]:
import requests, IPython.display as ipd, os

OUTPUT_DIR = "/workspace/tts_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

texts = [
    "Welcome to the AMD ROCm developer platform.",
    "Chatterbox TTS delivers high quality voice synthesis.",
    "This notebook runs entirely on AMD GPU hardware.",
]
VOICE = "Ryan.wav"

for i, text in enumerate(texts):
    r = requests.post("http://localhost:8004/v1/audio/speech", json={
        "model": "tts-1", "input": text, "voice": VOICE,
    }, timeout=120)

    if r.status_code == 200:
        path = os.path.join(OUTPUT_DIR, f"clip_{i+1:02d}.wav")
        with open(path, "wb") as f:
            f.write(r.content)
        print(f"[{i+1}/{len(texts)}] {path}（{len(r.content):,} 字节）")
        ipd.display(ipd.Audio(r.content))
    else:
        print(f"[{i+1}] ❌ 错误：{r.text}")

print(f"\n所有文件已保存至 {OUTPUT_DIR}")

## 第十一步 — GPU 显存监控

In [ ]:
import subprocess, torch

print("=" * 50)
print("PyTorch 显存统计")
print("=" * 50)
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved  = torch.cuda.memory_reserved(0)  / 1024**3
    total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"已分配：{allocated:.2f} GB")
    print(f"已预留：{reserved:.2f} GB")
    print(f"总显存：{total:.2f} GB")
    print(f"剩余：  {total - reserved:.2f} GB")

print("\nrocm-smi 显存信息：")
smi = subprocess.run(["rocm-smi", "--showmeminfo", "vram"], capture_output=True, text=True)
print(smi.stdout or smi.stderr)

## 第十二步 — 停止服务器

In [ ]:
import subprocess
result = subprocess.run(["pkill", "-f", "server.py"], capture_output=True)
print("✅ 服务器已停止" if result.returncode == 0 else "服务器未在运行")

---

## 常见问题

| 问题 | 解决方法 |
|---|---|
| `ROCm available: False` | 检查 `rocminfo` 输出；确认用户在 `video,render` 组 |
| HuggingFace 下载失败 | 确认 `HF_ENDPOINT=https://hf-mirror.com` 已设置 |
| `chatterbox-v2` 安装失败 | 从本地电脑下载 zip 上传到 `/workspace/` 后重试 |
| `HIP error: invalid device function` | 在第六步 env 中取消 `HSA_OVERRIDE_GFX_VERSION=11.0.0` 的注释 |
| 端口 8004 外部无法访问 | 平台仅暴露 8888（Jupyter），使用本 Notebook 的 API 单元格即可 |
| 首次启动很慢 | 需下载约 2-3 GB 模型，后续启动直接读缓存 |

## 参考资源

- [Chatterbox TTS Server](https://github.com/devnen/Chatterbox-TTS-Server)
- [AMD ROCm AI 开发者中心](https://rocm.docs.amd.com/projects/ai-developer-hub/en/latest/)
- [AMD AI Playbooks](https://developer.amd.com/playbooks/)
- [ROCm 官方文档](https://rocm.docs.amd.com/)
- [Hello ROCm 中文教程](https://datawhalechina.github.io/hello-rocm/)
- [HuggingFace 镜像站](https://hf-mirror.com)